In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Locate the repository from either its root or notebooks/ directory.
ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "data" / "raw").is_dir()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Cannot find data/raw. Open Jupyter inside NetGuard-ML."
    )

DATA_DIR = ROOT / "data" / "raw"
RANDOM_STATE = 42

## load the Training Dataset and Testing Dataset

In [2]:
# Local filenames appear reversed.
# Our decision: larger file = development; smaller file = test.
development_path = DATA_DIR / "UNSW_NB15_testing-set.csv"
test_path = DATA_DIR / "UNSW_NB15_training-set.csv"

development_df = pd.read_csv(development_path)
test_df = pd.read_csv(test_path)

assert len(development_df) == 175_341
assert len(test_df) == 82_332
assert development_df.columns.equals(test_df.columns)

print("Development:", development_df.shape)
print("Test:", test_df.shape)

Development: (175341, 45)
Test: (82332, 45)


## Seperate input and target

In [3]:
TARGET = "label"
CATEGORICAL_FEATURES = ["proto", "service", "state"]

FEATURES = [
    column
    for column in development_df.columns
    if column not in ["id", "label", "attack_cat"]
]

NUMERICAL_FEATURES = [
    column
    for column in FEATURES
    if column not in CATEGORICAL_FEATURES
]

X_development = development_df[FEATURES].copy()
y_development = development_df[TARGET].copy()

X_test = test_df[FEATURES].copy()

assert len(FEATURES) == 42
assert len(NUMERICAL_FEATURES) == 39

print("Numerical features:", len(NUMERICAL_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Total input features:", len(FEATURES))

Numerical features: 39
Categorical features: 3
Total input features: 42


## Give identical input rows the same group number

In [4]:
# Check inputs before grouping.
for name, frame in [
    ("development", X_development),
    ("test", X_test),
]:
    assert not frame.isna().any().any(), f"{name}: missing values"
    assert np.isfinite(
        frame[NUMERICAL_FEATURES].to_numpy(dtype=float)
    ).all(), f"{name}: infinite values"

# Combine inputs only to identify matching patterns across both files.
# This does NOT combine the files for model training.
combined_inputs = pd.concat(
    [X_development, X_test],
    ignore_index=True,
)

# Each distinct combination of the 42 features gets a group number.
pattern_index = pd.MultiIndex.from_frame(combined_inputs)
group_ids, _ = pd.factorize(pattern_index, sort=False)

development_groups = group_ids[:len(X_development)]
test_groups = group_ids[len(X_development):]

print("Development rows:", len(X_development))
print("Unique development patterns:", len(np.unique(development_groups)))

Development rows: 175341
Unique development patterns: 101040


## Split the development data by group

In [ ]:
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Use one fixed fold for validation and the other four for training.
train_indices, validation_indices = next(
    splitter.split(
        X_development,
        y_development,
        groups=development_groups,
    )
)

X_train = X_development.iloc[train_indices].copy()
y_train = y_development.iloc[train_indices].copy()

X_validation = X_development.iloc[validation_indices].copy()
y_validation = y_development.iloc[validation_indices].copy()

# Confirm that no identical input pattern crosses the boundary.
train_group_set = set(development_groups[train_indices])
validation_group_set = set(development_groups[validation_indices])

assert train_group_set.isdisjoint(validation_group_set)
assert len(X_train) + len(X_validation) == len(X_development)
assert set(y_train.unique()) == {0, 1}
assert set(y_validation.unique()) == {0, 1}

summary = pd.DataFrame({
    "partition": ["training", "validation"],
    "rows": [len(X_train), len(X_validation)],
    "development_share": [
        len(X_train) / len(X_development),
        len(X_validation) / len(X_development),
    ],
    "attack_share": [
        y_train.mean(),
        y_validation.mean(),
    ],
})

display(summary)

,partition,rows,development_share,attack_share
0,training,140272,0.799995,0.680621
1,validation,35069,0.200005,0.680630


In [6]:
class_balance = pd.DataFrame({
    "partition": ["development", "training", "validation"],
    "attack_percent": [
        y_development.mean() * 100,
        y_train.mean() * 100,
        y_validation.mean() * 100,
    ],
    "normal_percent": [
        (1 - y_development.mean()) * 100,
        (1 - y_train.mean()) * 100,
        (1 - y_validation.mean()) * 100,
    ],
})

display(class_balance.round(2))

,partition,attack_percent,normal_percent
0,development,68.06,31.94
1,training,68.06,31.94
2,validation,68.06,31.94


In [7]:
test_overlap_mask = np.isin(
    test_groups,
    np.unique(development_groups),
)

primary_test_mask = ~test_overlap_mask

assert set(test_groups[primary_test_mask]).isdisjoint(
    set(development_groups)
)

print("Full test rows:", len(X_test))
print("Overlapping test rows:", int(test_overlap_mask.sum()))
print("Primary test rows:", int(primary_test_mask.sum()))

Full test rows: 82332
Overlapping test rows: 8541
Primary test rows: 73791


In [8]:
import hashlib
import json

# Calculate a fingerprint of each source file.
def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# Record which files and feature definitions these assignments belong to.
split_metadata = {
    "development_file": development_path.name,
    "test_file": test_path.name,
    "development_sha256": file_sha256(development_path),
    "test_sha256": file_sha256(test_path),
    "features": FEATURES,
    "random_state": RANDOM_STATE,
    "split_method": "StratifiedGroupKFold: 5 folds, first fold validation",
}

split_directory = ROOT / "data" / "processed"
split_directory.mkdir(parents=True, exist_ok=True)

split_path = split_directory / "baseline_split_v1.npz"

# "xb" creates a new file and refuses to overwrite an existing split.
with split_path.open("xb") as file:
    np.savez_compressed(
        file,
        train_indices=train_indices,
        validation_indices=validation_indices,
        development_groups=development_groups,
        test_groups=test_groups,
        test_overlap_mask=test_overlap_mask,
        primary_test_mask=primary_test_mask,
        metadata=np.array(json.dumps(split_metadata)),
    )

print("Saved split:", split_path)

Saved split: /media/chanul/Coding Folders/NetGuard-ML/data/processed/baseline_split_v1.npz


## Data Preprocesing

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

lr_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            NUMERICAL_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

In [10]:
X_train_lr = lr_preprocessor.fit_transform(X_train)

X_validation_lr = lr_preprocessor.transform(X_validation)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_lr.shape)
print("Processed validation shape:", X_validation_lr.shape)

assert X_train_lr.shape[0] == len(y_train)
assert X_validation_lr.shape[0] == len(y_validation)
assert X_train_lr.shape[1] == X_validation_lr.shape[1]

Original training shape: (140272, 42)
Processed training shape: (140272, 193)
Processed validation shape: (35069, 193)


## Dummy Claasifier

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

# This model always predicts the most common training class.
dummy_model = DummyClassifier(strategy="most_frequent")

# Learn which class is most frequent in y_train.
dummy_model.fit(X_train_lr, y_train)

# Produce class predictions for validation rows.
dummy_predictions = dummy_model.predict(X_validation_lr)

# Produce the predicted probability of Attack (class 1).
attack_class_position = list(dummy_model.classes_).index(1)

dummy_attack_probabilities = dummy_model.predict_proba(
    X_validation_lr
)[:, attack_class_position]

# Confusion matrix layout:
# [[true normal, false attack],
#  [missed attack, correctly detected attack]]
tn, fp, fn, tp = confusion_matrix(
    y_validation,
    dummy_predictions,
    labels=[0, 1],
).ravel()

dummy_results = {
    "model": "Dummy — most frequent",
    "accuracy": accuracy_score(
        y_validation,
        dummy_predictions,
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validation,
        dummy_predictions,
    ),
    "attack_precision": precision_score(
        y_validation,
        dummy_predictions,
        zero_division=0,
    ),
    "attack_recall": recall_score(
        y_validation,
        dummy_predictions,
        zero_division=0,
    ),
    "attack_f1": f1_score(
        y_validation,
        dummy_predictions,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_validation,
        dummy_attack_probabilities,
    ),
    "pr_auc": average_precision_score(
        y_validation,
        dummy_attack_probabilities,
    ),
    "true_negatives": tn,
    "false_positives": fp,
    "false_negatives": fn,
    "true_positives": tp,
}

display(pd.DataFrame([dummy_results]).round(4))

,model,accuracy,balanced_accuracy,attack_precision,attack_recall,attack_f1,roc_auc,pr_auc,true_negatives,false_positives,false_negatives,true_positives
0,Dummy — most frequent,0.6806,0.5,0.6806,1.0,0.81,0.5,0.6806,0,11200,0,23869


## Linear Regression Model

In [13]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    solver="lbfgs",
    C=1.0,
    max_iter=1000,
)

lr_model.fit(X_train_lr, y_train)

print("Optimization iterations:", lr_model.n_iter_)

Optimization iterations: [263]


## Fit the Validation Data

In [14]:
# Predict Normal (0) or Attack (1).
lr_predictions = lr_model.predict(X_validation_lr)

# Find the probability column corresponding to Attack.
attack_position = list(lr_model.classes_).index(1)

lr_attack_probabilities = lr_model.predict_proba(
    X_validation_lr
)[:, attack_position]

# Count correct predictions, false alarms, and missed attacks.
tn, fp, fn, tp = confusion_matrix(
    y_validation,
    lr_predictions,
    labels=[0, 1],
).ravel()

lr_results = {
    "model": "Logistic regression",
    "accuracy": accuracy_score(y_validation, lr_predictions),
    "balanced_accuracy": balanced_accuracy_score(
        y_validation, lr_predictions
    ),
    "attack_precision": precision_score(
        y_validation, lr_predictions, zero_division=0
    ),
    "attack_recall": recall_score(
        y_validation, lr_predictions, zero_division=0
    ),
    "attack_f1": f1_score(
        y_validation, lr_predictions, zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_validation, lr_attack_probabilities
    ),
    "average_precision": average_precision_score(
        y_validation, lr_attack_probabilities
    ),
    "true_negatives": tn,
    "false_positives": fp,
    "false_negatives": fn,
    "true_positives": tp,
}

# Correct the dummy metric's name without changing the original dictionary.
dummy_comparison = dummy_results.copy()
dummy_comparison["average_precision"] = dummy_comparison.pop("pr_auc")

comparison = pd.DataFrame([dummy_comparison, lr_results])

display(comparison.round(4))

,model,accuracy,balanced_accuracy,attack_precision,attack_recall,attack_f1,roc_auc,true_negatives,false_positives,false_negatives,true_positives,average_precision
0,Dummy — most frequent,0.6806,0.5000,0.6806,1.000,0.810,0.5000,0,11200,0,23869,0.6806
1,Logistic regression,0.9367,0.9077,0.9242,0.988,0.955,0.9844,9266,1934,286,23583,0.9918


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", "passthrough", NUMERICAL_FEATURES),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessing", rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=2,
            ),
        ),
    ]
)

In [17]:
rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](42,)","['dur','proto','service',...,'ct_src_ltm','ct_srv_dst','is_sm_ips_ports']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,42
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='pas

In [18]:
rf_predictions = rf_pipeline.predict(X_validation)

attack_position = list(rf_pipeline.classes_).index(1)

rf_attack_probabilities = rf_pipeline.predict_proba(
    X_validation
)[:, attack_position]

tn, fp, fn, tp = confusion_matrix(
    y_validation,
    rf_predictions,
    labels=[0, 1],
).ravel()

rf_results = {
    "model": "Random forest",
    "accuracy": accuracy_score(y_validation, rf_predictions),
    "balanced_accuracy": balanced_accuracy_score(
        y_validation, rf_predictions
    ),
    "attack_precision": precision_score(
        y_validation, rf_predictions, zero_division=0
    ),
    "attack_recall": recall_score(
        y_validation, rf_predictions, zero_division=0
    ),
    "attack_f1": f1_score(
        y_validation, rf_predictions, zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_validation, rf_attack_probabilities
    ),
    "average_precision": average_precision_score(
        y_validation, rf_attack_probabilities
    ),
    "true_negatives": tn,
    "false_positives": fp,
    "false_negatives": fn,
    "true_positives": tp,
}

comparison = pd.DataFrame([
    dummy_comparison,
    lr_results,
    rf_results,
])

display(comparison.round(4))

,model,accuracy,balanced_accuracy,attack_precision,attack_recall,attack_f1,roc_auc,true_negatives,false_positives,false_negatives,true_positives,average_precision
0,Dummy — most frequent,0.6806,0.5000,0.6806,1.0000,0.8100,0.5000,0,11200,0,23869,0.6806
1,Logistic regression,0.9367,0.9077,0.9242,0.9880,0.9550,0.9844,9266,1934,286,23583,0.9918
2,Random forest,0.9555,0.9445,0.9603,0.9749,0.9675,0.9924,10238,962,600,23269,0.9961


## Neural Network Implemntation

In [19]:
import tensorflow as tf
from tensorflow import keras
from scipy import sparse

# Control random initialization and training randomness.
keras.utils.set_random_seed(RANDOM_STATE)


def to_dense_float32(data):
    if sparse.issparse(data):
        return data.astype(np.float32).toarray()

    return np.asarray(data, dtype=np.float32)


X_train_nn = to_dense_float32(X_train_lr)
X_validation_nn = to_dense_float32(X_validation_lr)

y_train_nn = np.asarray(y_train, dtype=np.float32)
y_validation_nn = np.asarray(y_validation, dtype=np.float32)

print("TensorFlow version:", tf.__version__)
print("Training shape:", X_train_nn.shape)
print("Validation shape:", X_validation_nn.shape)

I0000 00:00:1788178431.072560   75623 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788178431.493035   75623 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788178433.845104   75623 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0
Training shape: (140272, 193)
Validation shape: (35069, 193)


In [20]:
nn_model = keras.Sequential([
    keras.Input(shape=(X_train_nn.shape[1],)),

    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(32, activation="relu"),

    keras.layers.Dense(1, activation="sigmoid"),
])

nn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
    ],
)

nn_model.summary()

E0000 00:00:1788178468.820437   75623 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,529 (56.75 KB)

 Trainable params: 14,529 (56.75 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

nn_history = nn_model.fit(
    X_train_nn,
    y_train_nn,
    validation_data=(X_validation_nn, y_validation_nn),
    epochs=100,
    batch_size=512,
    shuffle=True,
    callbacks=[early_stopping],
    verbose=2,
)

print("Epochs completed:", len(nn_history.history["loss"]))

best_epoch = int(np.argmin(nn_history.history["val_loss"])) + 1
print("Best validation-loss epoch:", best_epoch)

Epoch 1/100


W0000 00:00:1788178488.552899   75623 cpu_allocator_impl.cc:82] Allocation of 108289984 exceeds 10% of free system memory.


274/274 - 3s - 11ms/step - accuracy: 0.9201 - loss: 0.2094 - val_accuracy: 0.9393 - val_loss: 0.1325
Epoch 2/100
274/274 - 1s - 4ms/step - accuracy: 0.9378 - loss: 0.1288 - val_accuracy: 0.9412 - val_loss: 0.1243
Epoch 3/100
274/274 - 1s - 5ms/step - accuracy: 0.9399 - loss: 0.1230 - val_accuracy: 0.9420 - val_loss: 0.1213
Epoch 4/100
274/274 - 2s - 8ms/step - accuracy: 0.9410 - loss: 0.1202 - val_accuracy: 0.9420 - val_loss: 0.1199
Epoch 5/100
274/274 - 1s - 5ms/step - accuracy: 0.9421 - loss: 0.1184 - val_accuracy: 0.9423 - val_loss: 0.1188
Epoch 6/100
274/274 - 1s - 5ms/step - accuracy: 0.9423 - loss: 0.1172 - val_accuracy: 0.9435 - val_loss: 0.1179
Epoch 7/100
274/274 - 2s - 9ms/step - accuracy: 0.9429 - loss: 0.1162 - val_accuracy: 0.9440 - val_loss: 0.1170
Epoch 8/100
274/274 - 1s - 4ms/step - accuracy: 0.9432 - loss: 0.1154 - val_accuracy: 0.9444 - val_loss: 0.1164
Epoch 9/100
274/274 - 1s - 4ms/step - accuracy: 0.9435 - loss: 0.1146 - val_accuracy: 0.9445 - val_loss: 0.1159
Epo

In [23]:
# One sigmoid output: estimated Attack probability for each row.
nn_attack_probabilities = nn_model.predict(
    X_validation_nn,
    batch_size=512,
    verbose=0,
).ravel()

# Convert probabilities into labels using a fixed 0.5 threshold.
nn_predictions = (nn_attack_probabilities >= 0.5).astype(int)

In [25]:
tn, fp, fn, tp = confusion_matrix(
    y_validation,
    nn_predictions,
    labels=[0, 1],
).ravel()

nn_results = {
    "model": "Neural network",
    "accuracy": accuracy_score(y_validation, nn_predictions),
    "balanced_accuracy": balanced_accuracy_score(
        y_validation, nn_predictions
    ),
    "attack_precision": precision_score(
        y_validation, nn_predictions, zero_division=0
    ),
    "attack_recall": recall_score(
        y_validation, nn_predictions, zero_division=0
    ),
    "attack_f1": f1_score(
        y_validation, nn_predictions, zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_validation, nn_attack_probabilities
    ),
    "average_precision": average_precision_score(
        y_validation, nn_attack_probabilities
    ),
    "true_negatives": tn,
    "false_positives": fp,
    "false_negatives": fn,
    "true_positives": tp,
}

comparison = pd.DataFrame([
    dummy_comparison,
    lr_results,
    rf_results,
    nn_results,
])

display(comparison.round(4))

,model,accuracy,balanced_accuracy,attack_precision,attack_recall,attack_f1,roc_auc,true_negatives,false_positives,false_negatives,true_positives,average_precision
0,Dummy — most frequent,0.6806,0.5000,0.6806,1.0000,0.8100,0.5000,0,11200,0,23869,0.6806
1,Logistic regression,0.9367,0.9077,0.9242,0.9880,0.9550,0.9844,9266,1934,286,23583,0.9918
2,Random forest,0.9555,0.9445,0.9603,0.9749,0.9675,0.9924,10238,962,600,23269,0.9961
3,Neural network,0.9466,0.9301,0.9474,0.9757,0.9613,0.9902,9907,1293,580,23289,0.9953


In [26]:
# Predict both partitions using the existing fitted pipeline.
rf_train_predictions = rf_pipeline.predict(X_train)
rf_validation_predictions = rf_pipeline.predict(X_validation)


def summarize_predictions(actual_labels, predicted_labels):
    tn, fp, fn, tp = confusion_matrix(
        actual_labels,
        predicted_labels,
        labels=[0, 1],
    ).ravel()

    return {
        "rows": len(actual_labels),
        "accuracy": accuracy_score(
            actual_labels, predicted_labels
        ),
        "balanced_accuracy": balanced_accuracy_score(
            actual_labels, predicted_labels
        ),
        "attack_precision": precision_score(
            actual_labels, predicted_labels, zero_division=0
        ),
        "attack_recall": recall_score(
            actual_labels, predicted_labels, zero_division=0
        ),
        "attack_f1": f1_score(
            actual_labels, predicted_labels, zero_division=0
        ),
        "false_alarm_rate": fp / (fp + tn),
        "missed_attack_rate": fn / (fn + tp),
    }


rf_train_validation = pd.DataFrame(
    [
        summarize_predictions(y_train, rf_train_predictions),
        summarize_predictions(
            y_validation, rf_validation_predictions
        ),
    ],
    index=["training", "validation"],
)

display(rf_train_validation.round(4))

,rows,accuracy,balanced_accuracy,attack_precision,attack_recall,attack_f1,false_alarm_rate,missed_attack_rate
training,140272,0.9981,0.9976,0.9982,0.9991,0.9986,0.0038,0.0009
validation,35069,0.9555,0.9445,0.9603,0.9749,0.9675,0.0859,0.0251


In [27]:
from sklearn.base import clone

# Copy the pipeline's configuration, without its learned state.
rf_leaf5_pipeline = clone(rf_pipeline)

# Change only the forest's minimum leaf size.
rf_leaf5_pipeline.set_params(
    model__min_samples_leaf=5
)

# Train this new pipeline on the same training rows.
rf_leaf5_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](42,)","['dur','proto','service',...,'ct_src_ltm','ct_srv_dst','is_sm_ips_ports']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,42
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='pas

In [28]:
leaf5_train_predictions = rf_leaf5_pipeline.predict(X_train)

leaf5_validation_predictions = rf_leaf5_pipeline.predict(
    X_validation
)

leaf5_results = pd.DataFrame(
    [
        summarize_predictions(
            y_train,
            leaf5_train_predictions,
        ),
        summarize_predictions(
            y_validation,
            leaf5_validation_predictions,
        ),
    ],
    index=["training", "validation"],
)

experiment_comparison = pd.concat(
    {
        "Original forest": rf_train_validation,
        "Minimum leaf size 5": leaf5_results,
    },
    names=["model", "partition"],
)

display(experiment_comparison.round(4))

rows  accuracy  balanced_accuracy  \
model               partition                                         
Original forest     training    140272    0.9981             0.9976   
                    validation   35069    0.9555             0.9445   
Minimum leaf size 5 training    140272    0.9733             0.9618   
                    validation   35069    0.9531             0.9371   

                                attack_precision  attack_recall  attack_f1  \
model               partition                                                
Original forest     training              0.9982         0.9991     0.9986   
                    validation            0.9603         0.9749     0.9675   
Minimum leaf size 5 training              0.9679         0.9938     0.9807   
                    validation            0.9512         0.9816     0.9661   

                                false_alarm_rate  missed_attack_rate  
model               partition                                         
Original forest     training              0.0038              0.0009  
                    validation            0.0859              0.0251  
Minimum leaf size 5 training              0.0703              0.0062  
                    validation            0.1074              0.0184

## Put predictions beside the original validation records

In [29]:
# Copy the original validation records, including diagnostic metadata.
rf_error_analysis = development_df.iloc[
    validation_indices
].copy()

# Use the ORIGINAL forest's predictions, not the leaf-size-5 model.
rf_error_analysis["prediction"] = np.asarray(rf_predictions)

# Flag the two types of mistakes.
rf_error_analysis["missed_attack"] = (
    (rf_error_analysis["label"] == 1)
    & (rf_error_analysis["prediction"] == 0)
)

rf_error_analysis["false_alarm"] = (
    (rf_error_analysis["label"] == 0)
    & (rf_error_analysis["prediction"] == 1)
)

print("Missed attacks:", rf_error_analysis["missed_attack"].sum())
print("False alarms:", rf_error_analysis["false_alarm"].sum())

Missed attacks: 600
False alarms: 962


In [30]:
actual_attacks = rf_error_analysis[
    rf_error_analysis["label"] == 1
]

misses_by_category = (
    actual_attacks
    .groupby("attack_cat")
    .agg(
        attack_rows=("missed_attack", "size"),
        missed_attacks=("missed_attack", "sum"),
        miss_rate=("missed_attack", "mean"),
    )
)

misses_by_category["miss_percent"] = (
    misses_by_category["miss_rate"] * 100
)

display(
    misses_by_category[
        ["attack_rows", "missed_attacks", "miss_percent"]
    ].sort_values("missed_attacks", ascending=False)
)

,attack_rows,missed_attacks,miss_percent
attack_cat,,,
Fuzzers,3728,510,13.680258
Analysis,432,57,13.194444
Exploits,6744,26,0.385528
DoS,2523,3,0.118906
Shellcode,234,3,1.282051
Generic,7816,1,0.012794
Backdoor,361,0,0.000000
Reconnaissance,2008,0,0.000000
Worms,23,0,0.000000


## Find where normal traffic causes false alarms

In [31]:
actual_normal = rf_error_analysis[
    rf_error_analysis["label"] == 0
]

alarms_by_protocol_service = (
    actual_normal
    .groupby(["proto", "service"])
    .agg(
        normal_rows=("false_alarm", "size"),
        false_alarms=("false_alarm", "sum"),
        false_alarm_rate=("false_alarm", "mean"),
    )
)

alarms_by_protocol_service["false_alarm_percent"] = (
    alarms_by_protocol_service["false_alarm_rate"] * 100
)

display(
    alarms_by_protocol_service[
        ["normal_rows", "false_alarms", "false_alarm_percent"]
    ]
    .sort_values("false_alarms", ascending=False)
    .head(15)
)

normal_rows  false_alarms  false_alarm_percent
proto service                                                 
tcp   -                5468           574            10.497440
udp   -                1251           224            17.905675
tcp   http             1082           112            10.351201
      ftp               259            46            17.760618
      smtp              298             2             0.671141
      ftp-data          518             1             0.193050
udp   radius              1             1           100.000000
      dns              1479             1             0.067613
      snmp                1             1           100.000000
icmp  -                   2             0             0.000000
igmp  -                   2             0             0.000000
ospf  -                  14             0             0.000000
arp   -                 554             0             0.000000
tcp   ssh               271             0             0.000000

## check whether the missed attacks occur in those same traffic groups

In [32]:
# Select actual Fuzzers and Analysis attacks.
priority_attacks = rf_error_analysis.loc[
    (rf_error_analysis["label"] == 1)
    & rf_error_analysis["attack_cat"].isin(["Fuzzers", "Analysis"])
]

# Measure missed attacks within each protocol/service combination.
priority_breakdown = (
    priority_attacks
    .groupby(["attack_cat", "proto", "service"])
    .agg(
        attack_rows=("missed_attack", "size"),
        missed_attacks=("missed_attack", "sum"),
        miss_rate=("missed_attack", "mean"),
    )
)

priority_breakdown["miss_percent"] = (
    priority_breakdown["miss_rate"] * 100
)

display(
    priority_breakdown[
        ["attack_rows", "missed_attacks", "miss_percent"]
    ]
    .sort_values("missed_attacks", ascending=False)
    .head(15)
)

attack_rows  missed_attacks  miss_percent
attack_cat proto       service                                           
Fuzzers    tcp         -               2098             364     17.349857
           udp         -                904             117     12.942478
Analysis   tcp         http             123              57     46.341463
Fuzzers    tcp         ftp              119              18     15.126050
                       http             198              11      5.555556
Analysis   bbn-rcc     -                  1               0      0.000000
           ax.25       -                  1               0      0.000000
           argus       -                  2               0      0.000000
           chaos       -                  1               0      0.000000
           compaq-peer -                  2               0      0.000000
           cphb        -                  2               0      0.000000
           crtp        -                  1               0      0.000000
           crudp       -                  1               0      0.000000
           dcn         -                  1               0      0.000000
           bna         -                  1               0      0.000000

In [33]:
# Restrict the comparison to one protocol/service combination.
focus = rf_error_analysis.loc[
    (rf_error_analysis["proto"] == "tcp")
    & (rf_error_analysis["service"] == "-")
    & (
        (rf_error_analysis["label"] == 0)
        | (rf_error_analysis["attack_cat"] == "Fuzzers")
    )
].copy()

# Give each row a descriptive outcome label.
focus["outcome"] = np.select(
    [
        (focus["label"] == 0) & (focus["prediction"] == 0),
        (focus["label"] == 0) & (focus["prediction"] == 1),
        (focus["label"] == 1) & (focus["prediction"] == 0),
        (focus["label"] == 1) & (focus["prediction"] == 1),
    ],
    [
        "Normal: correct",
        "Normal: false alarm",
        "Fuzzers: missed",
        "Fuzzers: detected",
    ],
    default="Unexpected",
)

# Start with a small set of traffic-volume and duration features.
inspect_features = [
    "dur",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
]

display(focus.groupby("outcome").size().rename("rows"))

# Show the middle 50% of each feature distribution.
feature_comparison = (
    focus.groupby("outcome")[inspect_features]
    .quantile([0.25, 0.50, 0.75])
)

feature_comparison.index.names = ["outcome", "quantile"]

display(feature_comparison.round(3))

outcome
Fuzzers: detected      1734
Fuzzers: missed         364
Normal: correct        4894
Normal: false alarm     574
Name: rows, dtype: int64

dur  spkts  dpkts  sbytes   dbytes
outcome             quantile                                      
Fuzzers: detected   0.25      0.601   10.0    8.0   756.0    354.0
                    0.50      0.925   12.0    8.0  1058.0    612.0
                    0.75      1.429   14.0   10.0  2026.0    766.0
Fuzzers: missed     0.25      0.508   10.0    6.0   534.0    268.0
                    0.50      0.696   10.0    6.0   597.0    268.0
                    0.75      0.870   10.0    8.0  1292.5    354.0
Normal: correct     0.25      0.022   16.0   14.0  1540.0   1644.0
                    0.50      0.068   28.0   30.0  2646.0   3080.0
                    0.75      0.599   60.0   62.0  4014.0  39076.0
Normal: false alarm 0.25      0.580   10.0    6.0   588.0    268.0
                    0.50      0.883   10.0    8.0   933.0    354.0
                    0.75      1.352   12.0   10.0  2288.5    612.0

In [34]:
# Reshape numerical features into one column for grouped analysis.
numeric_long = focus.melt(
    id_vars="outcome",
    value_vars=NUMERICAL_FEATURES,
    var_name="feature",
    value_name="value",
)

numeric_summary = (
    numeric_long
    .groupby(["feature", "outcome"])
    .agg(
        q25=("value", lambda values: values.quantile(0.25)),
        median=("value", "median"),
        q75=("value", lambda values: values.quantile(0.75)),
        zero_percent=("value", lambda values: (values == 0).mean() * 100),
    )
)

# Show all features side by side, starting with their medians.
median_comparison = numeric_summary["median"].unstack("outcome")

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(median_comparison)

outcome,Fuzzers: detected,Fuzzers: missed,Normal: correct,Normal: false alarm
feature,,,,
ackdat,6.834650e-02,6.508750e-02,1.350000e-04,7.175600e-02
ct_dst_ltm,1.000000e+00,2.000000e+00,3.000000e+00,1.000000e+00
ct_dst_sport_ltm,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
ct_dst_src_ltm,3.000000e+00,4.000000e+00,3.000000e+00,3.000000e+00
ct_flw_http_mthd,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
ct_ftp_cmd,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
ct_src_dport_ltm,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
ct_src_ltm,2.000000e+00,2.000000e+00,3.000000e+00,2.000000e+00
ct_srv_dst,3.000000e+00,5.000000e+00,6.000000e+00,3.000000e+00


In [35]:
for feature in CATEGORICAL_FEATURES:
    print(f"\nFeature: {feature} — percentage within each outcome")

    category_percentages = pd.crosstab(
        focus[feature],
        focus["outcome"],
        normalize="columns",
    ) * 100

    display(category_percentages.round(2))


Feature: proto — percentage within each outcome


outcome,Fuzzers: detected,Fuzzers: missed,Normal: correct,Normal: false alarm
proto,,,,
tcp,100.0,100.0,100.0,100.0



Feature: service — percentage within each outcome


outcome,Fuzzers: detected,Fuzzers: missed,Normal: correct,Normal: false alarm
service,,,,
-,100.0,100.0,100.0,100.0



Feature: state — percentage within each outcome


outcome,Fuzzers: detected,Fuzzers: missed,Normal: correct,Normal: false alarm
state,,,,
CON,0.0,0.0,3.62,0.0
FIN,100.0,100.0,92.34,100.0
REQ,0.0,0.0,3.78,0.0
RST,0.0,0.0,0.27,0.0


In [36]:
# Count distinct actual labels for each complete 42-feature pattern.
labels_per_pattern = (
    focus.groupby(FEATURES, dropna=False)["label"]
    .transform("nunique")
)

focus["conflicting_pattern"] = labels_per_pattern > 1

conflict_summary = (
    focus.groupby("outcome")
    .agg(
        rows=("conflicting_pattern", "size"),
        rows_in_conflicting_patterns=("conflicting_pattern", "sum"),
    )
)

display(conflict_summary)

,rows,rows_in_conflicting_patterns
outcome,,
Fuzzers: detected,1734,0
Fuzzers: missed,364,0
Normal: correct,4894,0
Normal: false alarm,574,0


### Inspecting the proposed packet-direction feature

The Random Forest frequently confuses Fuzzer attacks with Normal traffic inside
the `tcp` and `service="-"` validation group. Previous error analysis showed
that these outcomes have similar individual feature values, although their
source and destination packet counts may have different relationships.

This step creates a candidate feature called `packet_direction_balance`:

\[
\text{packet direction balance}
=
\frac{\text{source packets} - \text{destination packets}}
{\text{source packets} + \text{destination packets} + 1}
\]

The feature measures packet-flow direction:

- A positive value means that there are more source packets.
- A negative value means that there are more destination packets.
- A value near zero means that packet counts are approximately balanced.
- The added `1` prevents division by zero.

This is a diagnostic step only. It does not retrain the model or change the
original dataset. The feature is calculated for each row in a copy of the
focused validation subset.

The quartiles of this feature are compared across correctly detected Fuzzers,
missed Fuzzers, correctly classified Normal traffic, and Normal false alarms.
If missed Fuzzers and Normal false alarms have meaningfully different
distributions, the feature may be tested in a controlled Random Forest
experiment. If their distributions overlap strongly, the feature will be
rejected.

In [40]:
def add_packet_direction_balance(X):
    X_engineered = X.copy()

    X_engineered["packet_direction_balance"] = (
        (X_engineered["spkts"] - X_engineered["dpkts"])
        / (X_engineered["spkts"] + X_engineered["dpkts"] + 1)
    )

    return X_engineered


X_train_fe = add_packet_direction_balance(X_train)
X_validation_fe = add_packet_direction_balance(X_validation)

NUMERICAL_FEATURES_FE = NUMERICAL_FEATURES + [
    "packet_direction_balance"
]

In [41]:
rf_fe_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            "passthrough",
            NUMERICAL_FEATURES_FE,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

rf_fe_pipeline = Pipeline(
    steps=[
        ("preprocessing", rf_fe_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=2,
            ),
        ),
    ]
)

rf_fe_pipeline.fit(X_train_fe, y_train)

rf_fe_train_predictions = rf_fe_pipeline.predict(X_train_fe)
rf_fe_validation_predictions = rf_fe_pipeline.predict(X_validation_fe)

In [42]:
rf_fe_results = pd.DataFrame(
    [
        summarize_predictions(
            y_train,
            rf_fe_train_predictions,
        ),
        summarize_predictions(
            y_validation,
            rf_fe_validation_predictions,
        ),
    ],
    index=["training", "validation"],
)

feature_experiment_comparison = pd.concat(
    {
        "Original forest": rf_train_validation,
        "Packet-balance forest": rf_fe_results,
    },
    names=["model", "partition"],
)

display(feature_experiment_comparison.round(4))

rows  accuracy  balanced_accuracy  \
model                 partition                                         
Original forest       training    140272    0.9981             0.9976   
                      validation   35069    0.9555             0.9445   
Packet-balance forest training    140272    0.9981             0.9976   
                      validation   35069    0.9542             0.9429   

                                  attack_precision  attack_recall  attack_f1  \
model                 partition                                                
Original forest       training              0.9982         0.9991     0.9986   
                      validation            0.9603         0.9749     0.9675   
Packet-balance forest training              0.9982         0.9991     0.9986   
                      validation            0.9592         0.9742     0.9666   

                                  false_alarm_rate  missed_attack_rate  
model                 partition                                         
Original forest       training              0.0038              0.0009  
                      validation            0.0859              0.0251  
Packet-balance forest training              0.0039              0.0009  
                      validation            0.0884              0.0258

#### Packet-direction feature experiment result

Adding `packet_direction_balance` did not improve the Random Forest.

Compared with the original model, the engineered model produced slightly lower
validation accuracy, balanced accuracy, attack precision, attack recall, and
attack F1. It also increased both the false-alarm rate and the missed-attack
rate.

The training results remained almost unchanged. Therefore, the engineered
feature did not reduce the observed training-validation gap.

This result suggests that the packet-direction feature does not add useful
information under the current Random Forest configuration, or that the model
already captures the relevant relationship through the original `spkts` and
`dpkts` features.

The feature is rejected. The original Random Forest remains the current
candidate model. The reserved test dataset was not used in this decision.

### Training-only correlation analysis for feature selection

The EDA identified several groups of highly correlated numerical features.
Correlation alone does not prove that a feature should be removed, so the
baseline retained all available features.

The original Random Forest achieved almost perfect training performance but
lower validation performance. This step therefore revisits numerical
redundancy as a possible source of unnecessary model complexity.

Spearman correlation is calculated using only the training partition because:

- Many numerical features are strongly skewed.
- Spearman correlation measures monotonic relationships without requiring a
  linear relationship.
- Validation and reserved test data must not influence feature-selection
  decisions.

Only numerical feature pairs with an absolute correlation of at least `0.90`
are displayed. This is a diagnostic step only: no features are removed and no
model is trained in this cell.

Correlation does not determine which feature to remove. Any removal must also
consider feature meaning, prediction-time availability, and validation
performance.

In [43]:
# Calculate training-only Spearman correlations.
training_spearman_correlation = (
    X_train[NUMERICAL_FEATURES]
    .corr(method="spearman")
)

# Keep only the upper triangle so every pair appears once.
upper_triangle_mask = np.triu(
    np.ones(training_spearman_correlation.shape, dtype=bool),
    k=1,
)

correlated_pairs = (
    training_spearman_correlation
    .where(upper_triangle_mask)
    .stack()
    .rename("spearman_correlation")
    .reset_index()
    .rename(
        columns={
            "level_0": "feature_1",
            "level_1": "feature_2",
        }
    )
)

correlated_pairs["absolute_correlation"] = (
    correlated_pairs["spearman_correlation"].abs()
)

strongly_correlated_pairs = (
    correlated_pairs
    .loc[
        correlated_pairs["absolute_correlation"] >= 0.90
    ]
    .sort_values(
        "absolute_correlation",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(strongly_correlated_pairs.round(4))

,feature_1,feature_2,spearman_correlation,absolute_correlation
0,is_ftp_login,ct_ftp_cmd,1.0000,1.0000
1,trans_depth,ct_flw_http_mthd,0.9996,0.9996
2,tcprtt,synack,0.9968,0.9968
3,tcprtt,ackdat,0.9937,0.9937
4,swin,dwin,0.9904,0.9904
5,dpkts,dbytes,0.9899,0.9899
6,synack,ackdat,0.9844,0.9844
7,dbytes,dmean,0.9632,0.9632
8,dload,dmean,0.9629,0.9629
9,rate,sload,0.9607,0.9607


In [44]:
calculated_tcprtt = (
    X_train["synack"] + X_train["ackdat"]
)

tcprtt_absolute_difference = (
    X_train["tcprtt"] - calculated_tcprtt
).abs()

tcprtt_matches_components = np.isclose(
    X_train["tcprtt"],
    calculated_tcprtt,
    rtol=1e-5,
    atol=1e-8,
)

tcprtt_redundancy_summary = pd.Series({
    "training_rows": len(X_train),
    "matching_rows": tcprtt_matches_components.sum(),
    "matching_percent": tcprtt_matches_components.mean() * 100,
    "median_absolute_difference": (
        tcprtt_absolute_difference.median()
    ),
    "maximum_absolute_difference": (
        tcprtt_absolute_difference.max()
    ),
})

display(tcprtt_redundancy_summary)

training_rows                  1.402720e+05
matching_rows                  1.402720e+05
matching_percent               1.000000e+02
median_absolute_difference     0.000000e+00
maximum_absolute_difference    4.440892e-16
dtype: float64

### Removing the redundant `tcprtt` feature

The training-only redundancy check confirmed that:

\[
\text{tcprtt} = \text{synack} + \text{ackdat}
\]

for all training rows. Therefore, `tcprtt` does not contain independent
numerical information when `synack` and `ackdat` are already available.

This experiment trains another Random Forest without `tcprtt`. Every other
part of the experiment remains unchanged:

- The same training and validation rows are used.
- `synack` and `ackdat` are retained.
- The same categorical features and encoding are used.
- The same Random Forest hyperparameters and random seed are used.
- The reserved test dataset is not used.

Although `tcprtt` is mathematically redundant, removing it is not guaranteed to
improve the model. A Random Forest uses one-feature-at-a-time decision splits,
so the precomputed total may still make useful boundaries easier to learn.

The reduced model will be compared with the original forest using training and
validation metrics. The feature will be removed only if validation performance
is maintained or improved without creating an undesirable error trade-off.

In [45]:
NUMERICAL_FEATURES_NO_TCPRTT = [
    feature
    for feature in NUMERICAL_FEATURES
    if feature != "tcprtt"
]

assert "tcprtt" not in NUMERICAL_FEATURES_NO_TCPRTT
assert len(NUMERICAL_FEATURES_NO_TCPRTT) == (
    len(NUMERICAL_FEATURES) - 1
)

rf_no_tcprtt_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            "passthrough",
            NUMERICAL_FEATURES_NO_TCPRTT,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

rf_no_tcprtt_pipeline = Pipeline(
    steps=[
        ("preprocessing", rf_no_tcprtt_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=2,
            ),
        ),
    ]
)

rf_no_tcprtt_pipeline.fit(X_train, y_train)

rf_no_tcprtt_train_predictions = (
    rf_no_tcprtt_pipeline.predict(X_train)
)

rf_no_tcprtt_validation_predictions = (
    rf_no_tcprtt_pipeline.predict(X_validation)
)

rf_no_tcprtt_results = pd.DataFrame(
    [
        summarize_predictions(
            y_train,
            rf_no_tcprtt_train_predictions,
        ),
        summarize_predictions(
            y_validation,
            rf_no_tcprtt_validation_predictions,
        ),
    ],
    index=["training", "validation"],
)

tcprtt_removal_comparison = pd.concat(
    {
        "Original forest": rf_train_validation,
        "Without tcprtt": rf_no_tcprtt_results,
    },
    names=["model", "partition"],
)

display(tcprtt_removal_comparison.round(4))

rows  accuracy  balanced_accuracy  \
model           partition                                         
Original forest training    140272    0.9981             0.9976   
                validation   35069    0.9555             0.9445   
Without tcprtt  training    140272    0.9981             0.9976   
                validation   35069    0.9541             0.9428   

                            attack_precision  attack_recall  attack_f1  \
model           partition                                                
Original forest training              0.9982         0.9991     0.9986   
                validation            0.9603         0.9749     0.9675   
Without tcprtt  training              0.9982         0.9991     0.9986   
                validation            0.9591         0.9742     0.9666   

                            false_alarm_rate  missed_attack_rate  
model           partition                                         
Original forest training              0.0038              0.0009  
                validation            0.0859              0.0251  
Without tcprtt  training              0.0038              0.0009  
                validation            0.0886              0.0258

#### `tcprtt` removal experiment result

Although `tcprtt` was confirmed to equal `synack + ackdat` for every training
row, removing it slightly reduced validation performance.

The reduced model produced lower accuracy, balanced accuracy, attack precision,
attack recall, and attack F1. Both the false-alarm rate and missed-attack rate
increased. Training performance remained unchanged, so removing the feature
also did not reduce the training-validation gap.

A mathematically redundant feature can still be useful to a Random Forest
because its decision trees split on one feature at a time. The precomputed
`tcprtt` total may provide a useful decision boundary that would otherwise
require multiple splits using `synack` and `ackdat`.

The removal is therefore rejected, and `tcprtt` remains in the selected feature
set. Correlation alone was not sufficient evidence for removing the feature.
The original Random Forest remains the current candidate model, and the
reserved test dataset was not used.

### Checking redundancy between FTP-related features

The EDA audit identified `is_ftp_login` and `ct_ftp_cmd` as the strongest
remaining redundancy candidate.

In the original EDA, these features had:

- Pearson correlation equal to `1.0`.
- Identical observed value frequencies.
- Identical zero percentages.
- Almost no univariate class separation.

However, perfect correlation does not necessarily mean that two columns contain
exactly the same row-level values. It can also represent a perfect linear or
monotonic relationship.

This step therefore compares the two columns row by row using only the training
partition. It also displays their unique values and a cross-tabulation showing
how the values occur together.

No feature is removed and no model is trained in this step. The validation and
reserved test partitions are not inspected.

In [46]:
ftp_values_match = X_train["is_ftp_login"].eq(
    X_train["ct_ftp_cmd"]
)

ftp_redundancy_summary = pd.Series({
    "training_rows": len(X_train),
    "matching_rows": ftp_values_match.sum(),
    "mismatching_rows": (~ftp_values_match).sum(),
    "matching_percent": ftp_values_match.mean() * 100,
    "is_ftp_login_unique_values": (
        X_train["is_ftp_login"].nunique()
    ),
    "ct_ftp_cmd_unique_values": (
        X_train["ct_ftp_cmd"].nunique()
    ),
})

display(ftp_redundancy_summary)

ftp_value_combinations = pd.crosstab(
    X_train["is_ftp_login"],
    X_train["ct_ftp_cmd"],
    rownames=["is_ftp_login"],
    colnames=["ct_ftp_cmd"],
    dropna=False,
)

display(ftp_value_combinations)

training_rows                 140272.0
matching_rows                 140272.0
mismatching_rows                   0.0
matching_percent                 100.0
is_ftp_login_unique_values         4.0
ct_ftp_cmd_unique_values           4.0
dtype: float64

ct_ftp_cmd,0,1,2,4
is_ftp_login,,,,
0,138241,0,0,0
1,0,2019,0,0
2,0,0,6,0
4,0,0,0,6


### Removing the duplicated `ct_ftp_cmd` feature

The training-only equality check confirmed that `is_ftp_login` and
`ct_ftp_cmd` contain exactly the same value in all 140,272 training rows.

Both features have the same four observed values (`0`, `1`, `2`, and `4`), and
the cross-tabulation contains no mismatching combinations. Therefore,
`ct_ftp_cmd` provides no additional information in the training partition when
`is_ftp_login` is retained.

This experiment trains a separate Random Forest without `ct_ftp_cmd`. All other
conditions remain unchanged:

- The same training and validation partitions are used.
- `is_ftp_login` remains available.
- The same categorical preprocessing is used.
- The same Random Forest parameters and random seed are used.
- The original model is not modified.
- The reserved test dataset is not used.

Even exact duplicate removal can slightly change a Random Forest because
removing a column changes the random feature subsets available to its trees.
The feature will be removed only if validation performance is maintained or
improved without an undesirable increase in missed attacks or false alarms.

In [48]:
NUMERICAL_FEATURES_NO_FTP_CMD = [
    feature
    for feature in NUMERICAL_FEATURES
    if feature != "ct_ftp_cmd"
]

assert "ct_ftp_cmd" not in NUMERICAL_FEATURES_NO_FTP_CMD
assert "is_ftp_login" in NUMERICAL_FEATURES_NO_FTP_CMD
assert len(NUMERICAL_FEATURES_NO_FTP_CMD) == (
    len(NUMERICAL_FEATURES) - 1
)

rf_no_ftp_cmd_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            "passthrough",
            NUMERICAL_FEATURES_NO_FTP_CMD,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

rf_no_ftp_cmd_pipeline = Pipeline(
    steps=[
        ("preprocessing", rf_no_ftp_cmd_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=2,
            ),
        ),
    ]
)

rf_no_ftp_cmd_pipeline.fit(X_train, y_train)

rf_no_ftp_cmd_train_predictions = (
    rf_no_ftp_cmd_pipeline.predict(X_train)
)

rf_no_ftp_cmd_validation_predictions = (
    rf_no_ftp_cmd_pipeline.predict(X_validation)
)

rf_no_ftp_cmd_results = pd.DataFrame(
    [
        summarize_predictions(
            y_train,
            rf_no_ftp_cmd_train_predictions,
        ),
        summarize_predictions(
            y_validation,
            rf_no_ftp_cmd_validation_predictions,
        ),
    ],
    index=["training", "validation"],
)

ftp_removal_comparison = pd.concat(
    {
        "Original forest": rf_train_validation,
        "Without ct_ftp_cmd": rf_no_ftp_cmd_results,
    },
    names=["model", "partition"],
)

display(ftp_removal_comparison.round(4))

rows  accuracy  balanced_accuracy  \
model              partition                                         
Original forest    training    140272    0.9981             0.9976   
                   validation   35069    0.9555             0.9445   
Without ct_ftp_cmd training    140272    0.9981             0.9976   
                   validation   35069    0.9544             0.9432   

                               attack_precision  attack_recall  attack_f1  \
model              partition                                                
Original forest    training              0.9982         0.9991     0.9986   
                   validation            0.9603         0.9749     0.9675   
Without ct_ftp_cmd training              0.9982         0.9991     0.9986   
                   validation            0.9595         0.9741     0.9667   

                               false_alarm_rate  missed_attack_rate  
model              partition                                         
Original forest    training              0.0038              0.0009  
                   validation            0.0859              0.0251  
Without ct_ftp_cmd training              0.0038              0.0009  
                   validation            0.0877              0.0259

#### `ct_ftp_cmd` removal experiment result

The training-only analysis confirmed that `ct_ftp_cmd` and `is_ftp_login`
contain identical values in all 140,272 training rows.

However, removing `ct_ftp_cmd` slightly reduced every reported validation
metric. Attack precision, recall, F1, balanced accuracy, and overall accuracy
decreased. Both the false-alarm rate and missed-attack rate increased.

Although the two columns contain the same training information, removing one
changes the random subsets of features considered by the Random Forest's
trees. Therefore, the fitted forest can change even when the removed feature
is mathematically duplicated by another feature.

According to the predefined validation decision rule, the removal is rejected.
The original Random Forest and its original feature set remain selected.

This result does not establish that duplicated features are universally useful.
It only shows that the current experiment provides no evidence that removing
`ct_ftp_cmd` improves this model. The reserved test dataset was not used.

### Selecting the Random Forest decision threshold

The original Random Forest remains the selected model because the controlled
feature-engineering and feature-removal experiments did not improve validation
performance.

The model produces an Attack probability for every network record. The default
classification threshold is `0.50`:

- Probability greater than or equal to the threshold → Attack (`1`)
- Probability below the threshold → Normal (`0`)

Changing the threshold does not retrain the model.

A lower threshold usually detects more attacks but creates more false alarms.
A higher threshold usually reduces false alarms but misses more attacks.

This step compares a predefined range of thresholds using only validation data.
The reserved test dataset remains untouched. The threshold will be selected
using balanced accuracy, attack precision, attack recall, false-alarm rate, and
missed-attack rate rather than accuracy alone.

### Correcting probability tie handling

The first threshold table classified a row as Attack when its predicted Attack
probability was greater than or equal to the threshold.

At threshold `0.50`, this differs from the Random Forest's default `predict()`
behaviour when both classes receive equal probability. Scikit-learn selects the
first class (`0`, Normal) during an exact tie, while the explicit `>= 0.50` rule
selects Attack.

Because the Random Forest contains 100 trees, probabilities exactly equal to
candidate thresholds can occur. The threshold comparison is therefore
recalculated using a strict greater-than rule.

With this rule, threshold `0.50` should reproduce the original Random Forest
predictions. This ensures that threshold alternatives are compared against the
actual baseline consistently.

In [ ]:
# Obtain Attack probabilities from the original Random Forest.
rf_attack_position = list(
    rf_pipeline.classes_
).index(1)

rf_validation_attack_probabilities = (
    rf_pipeline.predict_proba(X_validation)
    [:, rf_attack_position]
)

candidate_thresholds = np.round(
    np.arange(0.30, 0.701, 0.05),
    2,
)

threshold_results = []

for threshold in candidate_thresholds:
    threshold_predictions = (
        rf_validation_attack_probabilities >= threshold
    ).astype(int)

    threshold_metrics = summarize_predictions(
        y_validation,
        threshold_predictions,
    )

    threshold_results.append({
        "threshold": threshold,
        **threshold_metrics,
    })

threshold_comparison = pd.DataFrame(
    threshold_results
)

display(threshold_comparison.round(4))

,threshold,rows,accuracy,balanced_accuracy,attack_precision,attack_recall,attack_f1,false_alarm_rate,missed_attack_rate
0,0.30,35069,0.9465,0.9183,0.9301,0.9963,0.9620,0.1596,0.0037
1,0.35,35069,0.9491,0.9244,0.9362,0.9929,0.9637,0.1442,0.0071
2,0.40,35069,0.9510,0.9303,0.9431,0.9875,0.9648,0.1269,0.0125
3,0.45,35069,0.9535,0.9371,0.9508,0.9826,0.9664,0.1083,0.0174
4,0.50,35069,0.9552,0.9433,0.9588,0.9762,0.9674,0.0895,0.0238
5,0.55,35069,0.9538,0.9461,0.9648,0.9674,0.9661,0.0752,0.0326
6,0.60,35069,0.9527,0.9491,0.9710,0.9592,0.9651,0.0611,0.0408
7,0.65,35069,0.9492,0.9497,0.9764,0.9484,0.9622,0.0489,0.0516
8,0.70,35069,0.9448,0.9492,0.9810,0.9370,0.9585,0.0387,0.0630


In [50]:
threshold_results = []

for threshold in candidate_thresholds:
    # Strict > assigns exact threshold ties to Normal.
    threshold_predictions = (
        rf_validation_attack_probabilities > threshold
    ).astype(int)

    threshold_metrics = summarize_predictions(
        y_validation,
        threshold_predictions,
    )

    threshold_results.append({
        "threshold": threshold,
        **threshold_metrics,
    })

threshold_comparison = pd.DataFrame(
    threshold_results
)

display(threshold_comparison.round(4))

,threshold,rows,accuracy,balanced_accuracy,attack_precision,attack_recall,attack_f1,false_alarm_rate,missed_attack_rate
0,0.30,35069,0.9470,0.9194,0.9312,0.9957,0.9623,0.1569,0.0043
1,0.35,35069,0.9496,0.9257,0.9377,0.9919,0.9640,0.1404,0.0081
2,0.40,35069,0.9518,0.9318,0.9446,0.9870,0.9653,0.1233,0.0130
3,0.45,35069,0.9539,0.9383,0.9522,0.9816,0.9667,0.1051,0.0184
4,0.50,35069,0.9555,0.9445,0.9603,0.9749,0.9675,0.0859,0.0251
5,0.55,35069,0.9535,0.9463,0.9655,0.9661,0.9658,0.0736,0.0339
6,0.60,35069,0.9524,0.9495,0.9721,0.9576,0.9648,0.0587,0.0424
7,0.65,35069,0.9489,0.9500,0.9772,0.9470,0.9619,0.0471,0.0530
8,0.70,35069,0.9439,0.9490,0.9819,0.9349,0.9578,0.0368,0.0651


In [51]:
threshold_050_predictions = (
    rf_validation_attack_probabilities > 0.50
).astype(int)

assert np.array_equal(
    threshold_050_predictions,
    rf_pipeline.predict(X_validation),
)

print("Threshold 0.50 matches the original predictions.")

Threshold 0.50 matches the original predictions.


### Selected decision threshold

The decision threshold is fixed at `0.50`.

Among the evaluated thresholds, `0.50` produced the highest validation accuracy
and Attack F1 while maintaining strong Attack recall and precision. It also
reproduces the original Random Forest's default predictions when exact
probability ties are assigned to the Normal class.

Higher thresholds reduced false alarms but caused a substantial increase in
missed attacks. For example, threshold `0.65` produced the highest balanced
accuracy but increased the missed-attack rate from `2.51%` to `5.30%`.

Lower thresholds detected more attacks but produced considerably more false
alarms. No deployment-specific cost information currently justifies accepting
that trade-off.

The selected prediction rule is:

- Attack probability greater than `0.50` → Attack
- Attack probability equal to or below `0.50` → Normal

The model configuration, feature set, preprocessing, and decision threshold are
now locked. The reserved test dataset was not used for threshold selection.

In [52]:
SELECTED_THRESHOLD = 0.50

selected_validation_predictions = (
    rf_validation_attack_probabilities
    > SELECTED_THRESHOLD
).astype(int)

# Confirm that the locked rule reproduces the original model.
assert np.array_equal(
    selected_validation_predictions,
    rf_pipeline.predict(X_validation),
)

selected_threshold_result = pd.DataFrame([
    {
        "threshold": SELECTED_THRESHOLD,
        **summarize_predictions(
            y_validation,
            selected_validation_predictions,
        ),
    }
])

display(selected_threshold_result.round(4))

,threshold,rows,accuracy,balanced_accuracy,attack_precision,attack_recall,attack_f1,false_alarm_rate,missed_attack_rate
0,0.5,35069,0.9555,0.9445,0.9603,0.9749,0.9675,0.0859,0.0251


## Final primary-test evaluation

Model selection is now complete. The original Random Forest, original feature
set, preprocessing configuration, and decision threshold of `0.50` have been
locked using the development training and validation partitions.

The final model is refitted using the complete 175,341-row development dataset.
This allows the model to use both the previous training and validation rows now
that all development decisions have been completed.

Final evaluation is performed only on the 73,791 primary test rows whose exact
input patterns do not occur anywhere in the development dataset. This provides
a more realistic measurement of generalization to previously unseen input
patterns.

The remaining 8,541 overlapping test rows are excluded from the primary result
and may later be reported separately as supplementary analysis.

The test labels are used only to calculate final metrics. They are not used for
fitting, preprocessing, feature selection, threshold selection, or model
modification.

After viewing this result, the model must not be modified based on primary-test
performance. Any later improvement would require a new development protocol
and a new independent test set.

In [53]:
# Prepare the complete development target.
y_development = (
    development_df[TARGET]
    .astype(int)
    .copy()
)

# Prepare the reserved test target.
y_test = (
    test_df[TARGET]
    .astype(int)
    .copy()
)

# Select only test rows with no input-pattern overlap
# with the complete development dataset.
primary_test_positions = np.flatnonzero(
    primary_test_mask
)

X_primary_test = (
    X_test
    .iloc[primary_test_positions]
    .copy()
)

y_primary_test = (
    y_test
    .iloc[primary_test_positions]
    .copy()
)

assert len(X_development) == 175341
assert len(X_primary_test) == 73791
assert len(X_primary_test) == len(y_primary_test)
assert not test_overlap_mask[
    primary_test_positions
].any()

print("Development rows:", len(X_development))
print("Primary test rows:", len(X_primary_test))

Development rows: 175341
Primary test rows: 73791


### Fitting and evaluating the locked final model

A new Random Forest pipeline is created using the locked configuration. The
preprocessor and model are fitted from scratch using the complete development
dataset.

The fitted model produces Attack probabilities for the primary test rows.
Probabilities strictly greater than `0.50` are classified as Attack. Exact
`0.50` ties are classified as Normal, matching the validation threshold rule.

The reported metrics include overall and class-balanced performance, Attack
precision, recall and F1, false alarms, missed attacks, ROC-AUC, and average
precision.

In [54]:
final_rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            "passthrough",
            NUMERICAL_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

final_rf_pipeline = Pipeline(
    steps=[
        ("preprocessing", final_rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=2,
            ),
        ),
    ]
)

# Refit using all development rows.
final_rf_pipeline.fit(
    X_development,
    y_development,
)

final_attack_position = list(
    final_rf_pipeline.classes_
).index(1)

final_primary_attack_probabilities = (
    final_rf_pipeline.predict_proba(X_primary_test)
    [:, final_attack_position]
)

final_primary_predictions = (
    final_primary_attack_probabilities
    > SELECTED_THRESHOLD
).astype(int)

final_primary_metrics = {
    "model": "Final Random Forest",
    "partition": "Primary non-overlapping test",
    **summarize_predictions(
        y_primary_test,
        final_primary_predictions,
    ),
    "roc_auc": roc_auc_score(
        y_primary_test,
        final_primary_attack_probabilities,
    ),
    "average_precision": average_precision_score(
        y_primary_test,
        final_primary_attack_probabilities,
    ),
}

final_primary_result = pd.DataFrame([
    final_primary_metrics
])

display(final_primary_result.round(4))

,model,partition,rows,accuracy,balanced_accuracy,attack_precision,attack_recall,attack_f1,false_alarm_rate,missed_attack_rate,roc_auc,average_precision
0,Final Random Forest,Primary non-overlapping test,73791,0.8635,0.8589,0.7997,0.983,0.8819,0.2652,0.017,0.9763,0.9777


### Final primary-test result

The locked Random Forest was evaluated on 73,791 test rows whose exact input
patterns did not occur in the development dataset.

The model achieved `86.35%` accuracy, `85.89%` balanced accuracy, `98.30%`
Attack recall, and `88.19%` Attack F1. ROC-AUC was `97.63%`, and average
precision was `97.77%`.

The most important weakness is the false-alarm rate. The model incorrectly
classified `26.52%` of actual Normal records as attacks. Consequently, Attack
precision decreased to `79.97%`.

The missed-attack rate was only `1.70%`, meaning the model detected most actual
attacks. Therefore, the final model favours attack detection at the cost of
producing many Normal-traffic alerts.

Performance was substantially lower than development-validation performance,
especially for accuracy, balanced accuracy, precision, F1, and false-alarm
rate. This indicates a generalization or distribution-shift problem between
development traffic and the non-overlapping primary test subset.

The high ROC-AUC and average precision indicate that probability ranking
remains strong, but the selected operating threshold does not produce the same
error balance on the primary test distribution.

This primary test result is final. The feature set, model parameters, or
threshold will not be changed using this test result. Any improved model must
be developed as a new version using a revised validation protocol and evaluated
on a new independent test set.

In [55]:
final_tn, final_fp, final_fn, final_tp = confusion_matrix(
    y_primary_test,
    final_primary_predictions,
    labels=[0, 1],
).ravel()

final_confusion_summary = pd.Series({
    "actual_normal_rows": final_tn + final_fp,
    "actual_attack_rows": final_fn + final_tp,
    "true_negatives": final_tn,
    "false_positives": final_fp,
    "false_negatives": final_fn,
    "true_positives": final_tp,
})

display(final_confusion_summary)

actual_normal_rows    35528
actual_attack_rows    38263
true_negatives        26106
false_positives        9422
false_negatives         650
true_positives        37613
dtype: int64

## Final primary-test error analysis

The final model detected most attacks but produced a high false-alarm rate on
the non-overlapping primary test subset.

This post-test analysis identifies:

1. Which attack categories contain the 650 missed attacks.
2. Which protocol and service groups contain the 9,422 false alarms.

The analysis uses the already locked predictions. It does not retrain the
model, change the threshold, select features, or modify preprocessing.

Because the primary test labels have now been examined, these findings are
reported as Version 1 limitations only. They must not be used to tune Version 1
while continuing to describe this dataset as an untouched test set.

In [56]:
final_error_analysis = (
    test_df
    .iloc[primary_test_positions]
    .copy()
)

final_error_analysis["prediction"] = (
    final_primary_predictions
)

final_error_analysis["attack_probability"] = (
    final_primary_attack_probabilities
)

final_error_analysis["missed_attack"] = (
    (final_error_analysis[TARGET] == 1)
    & (final_error_analysis["prediction"] == 0)
)

final_error_analysis["false_alarm"] = (
    (final_error_analysis[TARGET] == 0)
    & (final_error_analysis["prediction"] == 1)
)

assert final_error_analysis["missed_attack"].sum() == 650
assert final_error_analysis["false_alarm"].sum() == 9422

In [57]:
primary_actual_attacks = final_error_analysis.loc[
    final_error_analysis[TARGET] == 1
]

final_misses_by_category = (
    primary_actual_attacks
    .groupby("attack_cat", dropna=False)
    .agg(
        attack_rows=(TARGET, "size"),
        missed_attacks=("missed_attack", "sum"),
        missed_attack_rate=("missed_attack", "mean"),
    )
    .sort_values(
        ["missed_attacks", "missed_attack_rate"],
        ascending=False,
    )
)

final_misses_by_category["miss_percent"] = (
    final_misses_by_category["missed_attack_rate"] * 100
)

display(
    final_misses_by_category[
        [
            "attack_rows",
            "missed_attacks",
            "miss_percent",
        ]
    ].round(4)
)

,attack_rows,missed_attacks,miss_percent
attack_cat,,,
Fuzzers,5801,596,10.2741
Exploits,9330,47,0.5038
Shellcode,365,3,0.8219
DoS,2085,3,0.1439
Reconnaissance,2358,1,0.0424
Analysis,552,0,0.0000
Backdoor,579,0,0.0000
Generic,17149,0,0.0000
Worms,44,0,0.0000


In [58]:
primary_actual_normal = final_error_analysis.loc[
    final_error_analysis[TARGET] == 0
]

final_false_alarms_by_traffic = (
    primary_actual_normal
    .groupby(
        ["proto", "service"],
        dropna=False,
    )
    .agg(
        normal_rows=(TARGET, "size"),
        false_positives=("false_alarm", "sum"),
        false_alarm_rate=("false_alarm", "mean"),
    )
    .sort_values(
        ["false_positives", "false_alarm_rate"],
        ascending=False,
    )
)

final_false_alarms_by_traffic["false_alarm_percent"] = (
    final_false_alarms_by_traffic[
        "false_alarm_rate"
    ] * 100
)

display(
    final_false_alarms_by_traffic[
        [
            "normal_rows",
            "false_positives",
            "false_alarm_percent",
        ]
    ].head(20).round(4)
)

normal_rows  false_positives  false_alarm_percent
proto service                                                    
tcp   -               21209             5628              26.5359
udp   -                4426             2224              50.2485
tcp   http             4013             1248              31.0989
      ftp               758              311              41.0290
igmp  -                  28                5              17.8571
udp   dns              3068                4               0.1304
      radius              2                2             100.0000
arp   -                 202                0               0.0000
ospf  -                  38                0               0.0000
tcp   ftp-data          949                0               0.0000
      smtp              635                0               0.0000
      ssh               200                0               0.0000